# Generate embeddings for the Odia QA dataset

This notebook embeds each row as `query: Question: ...\nContext: ...` with `intfloat/multilingual-e5-base`. The answer is deliberately excluded. The E5 `query:` prefix follows the model's retrieval-style input convention; mean pooling uses the attention mask, followed by L2 normalization.

The filtered dataset was already preprocessed: the existing pipeline removes missing/blank required fields, excludes held-out passage overlap and conflicting duplicate answers, and filters unanswerable synthetic answers. This notebook still fills any residual missing question or paragraph with an empty string without dropping rows. The tokenizer truncates to 512 tokens, the model's supported input limit, so long paragraphs are handled safely while preserving the question and the beginning of the context.

Install dependencies in the notebook environment before running. For a standard environment:

```bash
python -m pip install pandas pyarrow torch transformers tqdm
```

On an Intel Mac (`x86_64`), use Python 3.12 or earlier. PyTorch's last Intel Mac wheel is 2.2.2, so create a compatible environment and pin Transformers:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install "torch==2.2.2" "transformers==4.49.0" pandas pyarrow tqdm
```


## Run in Google Colab

1. In Colab, choose **File → Upload notebook** and select this `.ipynb`.
2. Put `or_synthetic_filtered.parquet` in a Google Drive folder, then run this in a new cell:

```python
from google.colab import drive
drive.mount("/content/drive")
```

3. In the paths cell below, set `DATA_PATH` and `OUTPUT_PATH` to paths in that Drive folder (for example, `/content/drive/MyDrive/odia_embedding/or_synthetic_filtered.parquet` and `/content/drive/MyDrive/odia_embedding/odia_embeddings.pt`).
4. Choose **Runtime → Change runtime type → T4 GPU** if offered. The notebook detects the GPU automatically; CPU also works.

The dataset needs to be uploaded to Drive or Colab separately from the notebook. Saving the `.pt` output to Drive keeps it after the Colab runtime ends.


In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

MODEL_NAME = "intfloat/multilingual-e5-base"
MAX_LENGTH = 512

# Resolve paths whether the notebook is launched from this folder or the repo root.
HERE = Path.cwd()
if HERE.name != "embedding_generation":
    if (HERE / "hadl_courseproject" / "embedding_generation").is_dir():
        HERE = HERE / "hadl_courseproject" / "embedding_generation"
    elif (HERE / "embedding_generation").is_dir():
        HERE = HERE / "embedding_generation"

DATA_PATH = HERE.parent / "data_and_scripts" / "odia_synth_cleaned" / "or_synthetic_filtered.parquet"
OUTPUT_PATH = HERE / "odia_embeddings.pt"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BATCH_SIZE = 32 if DEVICE.type in {"cuda", "mps"} else 8

print(f"Input: {DATA_PATH}")
print(f"Device: {DEVICE}; batch size: {BATCH_SIZE}")


In [ ]:
# Keep the dataframe in source order; we never sort or filter rows here.
required_columns = ["paragraph", "question", "answer", "wiki_id", "title", "url"]
df = pd.read_parquet(DATA_PATH, columns=required_columns)
assert list(df.columns) == required_columns, f"Unexpected columns: {df.columns.tolist()}"
row_count = len(df)

# Missing text becomes empty text. Answers are loaded for schema validation only and never embedded.
questions = df["question"].fillna("").astype(str).tolist()
paragraphs = df["paragraph"].fillna("").astype(str).tolist()
assert len(questions) == len(paragraphs) == row_count

# answers = df["answer"].fillna("").astype(str).tolist()
# assert len(answers) == row_count

print(f"Dataset rows: {row_count:,}")


In [ ]:
print(f"Downloading/checking model files for {MODEL_NAME} (weights are about 1.11 GB)...")
MODEL_DIR = snapshot_download(repo_id=MODEL_NAME)
print("Model files are downloaded. Loading model into memory...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()
print(f"Model loaded on {DEVICE}. Starting embedding generation...")

# Preallocate on CPU to avoid holding both a list of all batches and a concatenated copy in memory.
embeddings = torch.empty((row_count, model.config.hidden_size), dtype=torch.float32)

with torch.inference_mode():
    for start in tqdm(range(0, row_count, BATCH_SIZE), desc="Embedding rows"):
        end = min(start + BATCH_SIZE, row_count)
        batch_texts = [ # For answers give : f"passage: {answers[i].strip()}"
            f"query: Question: {questions[i].strip()}\nContext: {paragraphs[i].strip()}" 
            for i in range(start, end)
        ]
        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        token_embeddings = model(**tokens).last_hidden_state

        # Attention-mask-aware mean pooling, then unit-length normalization.
        mask = tokens["attention_mask"].unsqueeze(-1).to(token_embeddings.dtype)
        pooled = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        pooled = F.normalize(pooled.float(), p=2, dim=1)
        embeddings[start:end] = pooled.cpu()

print("Embedding generation complete. Saving embeddings...")
torch.save(embeddings, OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}")


KeyboardInterrupt: 

In [ ]:
print("Embedding matrix shape:", tuple(embeddings.shape))
assert embeddings.shape[0] == row_count, (
    f"Expected {row_count} embeddings, found {embeddings.shape[0]}"
)
print(f"Verified: {embeddings.shape[0]:,} embeddings for {row_count:,} dataset rows")
